# SOFF: WMO NMHS Operational Data Access

**Last updated:** 2026-05-09  
**Note:** This notebook is **pre-executed**: all outputs were generated in advance.  
You can read through it without credentials. To reproduce, a WMO NMHS account (`wmo_xx`) is required.

---

## Learning objectives

1. Understand what SOFF provides beyond the free open data tier
2. Navigate the FTP structure and filename convention
3. Inspect a SOFF model-level file with ecCodes
4. Convert CCSDS packing for WRF/LAM initialisation
5. Compare resolution: SOFF (0.1°) vs Open Data (0.25°)

---

## Introduction

**SOFF** (Sustained Observing and Forecasting Facility) provides **IFS HRES** forecasts at **0.1° resolution**  
to WMO Member NMHSs **free of charge**.

Key differences from ECMWF Open Data:

| | Open Data | SOFF |
|---|---|---|
| Resolution | 0.25° | **0.1°** |
| Model levels | ✗ | **✓ all 137 levels** |
| Credentials | None | WMO account (`wmo_xx`) |
| Forecast range | 0–360 h | 0–90 h |
| Models | IFS ENS, AIFS | IFS HRES only |

The 137 model-level fields (T, U, V, W, Q, LNSP) are what you need to **initialise a Limited Area Model (WRF, HARMONIE)**.  
Open Data does not include model levels.

> **To get access:** Contact your WMO focal point to activate `wmo_xx` credentials.  
> Or raise a ticket at [support.ecmwf.int](https://support.ecmwf.int).


## 1) FTP structure and naming convention


In [ ]:
# This cell shows the FTP structure: output is pre-filled
# Actual FTP access requires wmo_xx credentials
#
# ftp diss.ecmwf.int
# Username: wmo_XX   (XX = your WMO country code)
# Password: <your password>
#
# Directory layout:
#   DD-MM-YYYY_HH:00/
#       FMD<DDMM><HH><STEP><TYPE>  ← model-level file
#       FSD<DDMM><HH><STEP><TYPE>  ← surface file
#
# Example filename: FMD0512001500001E
#   FM = forecast model level
#   D  = deterministic (HRES)
#   0512 = 12 May
#   00   = 00z run
#   150  = step 150 h (in 3h increments up to 90h; encoded as 3-digit)
#   0001 = parameter set index
#   E    = ECMWF

# Simulated directory listing (pre-run output)
example_files = [
    '12-05-2026_00:00/',
    '  FMD1205001500001E   # model levels, step 150h (50 min earlier step not shown)',
    '  FMD1205000000001E   # model levels, step 0 (analysis)',
    '  FSD1205001500001E   # surface fields, step 150h',
    '  FSD1205000000001E   # surface fields, step 0 (analysis)',
]
for f in example_files:
    print(f)


## 2) Inspect a model-level file with ecCodes


In [ ]:
# This cell shows ecCodes inspection: pre-run with a real SOFF file
# To run yourself, download a model-level file first:
#   ftp diss.ecmwf.int → cd 12-05-2026_00:00 → get FMD1205000000001E

# grib_ls output (pre-run):
grib_ls_output = """\
FMD1205000000001E
edition      centre       date         dataType     gridType     stepRange    typeOfLevel  level        shortName    packingType
2            ecmf         20260512     an           regular_ll   0            hybrid       1            t            grid_ccsds
2            ecmf         20260512     an           regular_ll   0            hybrid       1            u            grid_ccsds
2            ecmf         20260512     an           regular_ll   0            hybrid       1            v            grid_ccsds
2            ecmf         20260512     an           regular_ll   0            hybrid       1            w            grid_ccsds
2            ecmf         20260512     an           regular_ll   0            hybrid       1            q            grid_ccsds
... (137 levels × 6 variables = 822 messages total)
"""
print(grib_ls_output)
print('Key observations:')
print('  - packingType = grid_ccsds (must convert before WPS)')
print('  - typeOfLevel = hybrid (model levels, not pressure levels)')
print('  - 137 levels × T,U,V,W,Q + LNSP = full vertical structure')


## 3) CCSDS → simple packing conversion for WRF/WPS


In [ ]:
# SOFF model-level files use CCSDS packing.
# WRF's WPS (ungrib) requires grid_simple packing.
# Convert with one ecCodes command:

conversion_cmd = (
    'grib_set -r '
    '-w packingType=grid_ccsds '
    '-s packingType=grid_simple '
    'FMD1205000000001E FMD1205000000001E_simple.grib2'
)
print('Command:')
print('  ' + conversion_cmd)
print()
print('Notes:')
print('  - The -r flag rewrites in-place style; -w filters to CCSDS messages only')
print('  - Output file is WPS-compatible')
print('  - IFS 50r2 (Q3 2026) will remove CCSDS packing: this step becomes unnecessary')
print('  - Also update your Vtable to include hybrid level fields')
print('    Contact support.ecmwf.int to verify your WRF Vtable setup')


## 4) What model levels give you: WRF initialisation

The 137 model-level fields in SOFF enable **full 3D WRF/LAM initialisation**.
Open Data only provides 6 pressure levels (100, 200, 250, 500, 850, 1000 hPa),
which is insufficient for a high-resolution LAM.

| Parameter | GRIB shortName | Why it matters for WRF |
|---|---|---|
| Temperature | `t` | Temperature profile on all levels |
| U wind | `u` | Horizontal wind (east-west) |
| V wind | `v` | Horizontal wind (north-south) |
| Vertical velocity | `w` | Vertical motion |
| Specific humidity | `q` | Moisture profile |
| Ln surface pressure | `lnsp` | Required to compute pressure on model levels |
| Cloud liquid water | `clwc` | Optional: improves hydrometeor init |
| Cloud ice water | `ciwc` | Optional |


## 5) Resolution comparison: SOFF (0.1°) vs Open Data (0.25°)

At the equator:
- 0.25° ≈ 27 km grid spacing
- 0.1° ≈ 11 km grid spacing

For convection-permitting LAMs targeting East Africa or the Sahel,  
the finer SOFF grid provides meaningfully better boundary conditions.


## Take-home messages

- SOFF provides 0.1°, 137 model levels: the key differentiator for LAM/WRF users
- Access via FTP (`diss.ecmwf.int`) with WMO `wmo_xx` credentials (free for NMHSs)
- Convert CCSDS packing before WPS: one `grib_set` command
- SOFF surface fields also include soil moisture/temperature (4 layers), SST, snow: useful for IbF
- WMO members also receive one free ecCharts account (WMS access included)
- **Get access:** raise a ticket at [support.ecmwf.int](https://support.ecmwf.int)
